# ch05 Bonus 02：学习率调度器（LR Schedulers）

> 对照官方 `ch05/04_learning_rate_schedulers`

## 一句话

训练全程用固定学习率效果差。好的策略：先 **warmup**（小→大）避免初期发散，再 **cosine 衰减**（大→小）让训练后期稳定收敛。

## 为什么需要调度

- **初期**：参数随机初始化，大 lr 会让 loss 爆炸。需要 warmup 从小到大逐步升高。
- **后期**：接近最优解时，大 lr 会反复震荡。需要 cosine 衰减让步长变小，精细收敛。

这是现代 LLM 训练（GPT-3、Llama 等）的标准配置：`warmup + cosine decay`。

In [ ]:
import math
import torch


class CosineWithWarmup:
    """warmup 线性升 → cosine 余弦降到 min_lr。"""

    def __init__(self, optimizer, num_warmup, num_training, base_lr, min_lr=0.0):
        self.optimizer = optimizer
        self.num_warmup = num_warmup
        self.num_training = num_training
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.step_num = 0

    def step(self):
        self.step_num += 1
        if self.step_num < self.num_warmup:
            # warmup 阶段：线性从 0 升到 base_lr
            lr = self.base_lr * self.step_num / self.num_warmup
        else:
            # cosine 衰减阶段：从 base_lr 余弦降到 min_lr
            progress = (self.step_num - self.num_warmup) / (self.num_training - self.num_warmup)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        return lr

In [ ]:
# 模拟一次完整训练的 lr 曲线
params = [torch.randn(3, requires_grad=True)]
base_lr, warmup, total = 3e-4, 50, 500
opt = torch.optim.AdamW(params, lr=base_lr)
sched = CosineWithWarmup(opt, warmup, total, base_lr, min_lr=base_lr*0.1)

lrs = [sched.step() for _ in range(total)]
print(f"base_lr={base_lr}, warmup={warmup}步, total={total}步")
print(f"初始 lr:     {lrs[0]:.6f}")
print(f"warmup 顶点: {max(lrs):.6f}  (step {lrs.index(max(lrs))})")
print(f"结束 lr:     {lrs[-1]:.6f}")

# 文字版曲线示意
print("\nlr 曲线（每 50 步采样）：")
for i in range(0, total, 50):
    bar = "█" * int(lrs[i] / base_lr * 30)
    print(f"  step {i:3d} | {lrs[i]:.5f} {bar}")